In [0]:
def generate_employee_spark(cantidad):
    """
    Genera datos sintéticos de empleados con problemas de calidad (zona de aterrizaje sin limpiar).
    Datos diseñados para practicar limpieza, transformación y validación.
    """
    
    # Generamos el DataFrame base con IDs usando spark.range
    df_base = spark.range(1, cantidad + 1).withColumnRenamed("id", "employee_id")
    
    # Arrays de valores para selección aleatoria
    first_names = ["Juan", "Manuel", "Maria", "Pedro", "Jose", "Ana", "Carlos", "Laura", "Sofia", "Diego", 
                   "Luis", "Fernando", "Andres", "Camila", "Valentina", "Santiago", "Daniela", "Gabriela", 
                   "Javier", "Natalia", "Sebastian", "Valeria", "Isabella", "Alejandro", "Mariana", "Miguel", 
                   "Paula", "Ricardo", "Carolina", "Felipe", "Adriana", "Gustavo", "Monica", "Rodrigo", "Beatriz"]
    
    last_names = ["Rodriguez", "Gonzalez", "Perez", "Martinez", "Garcia", "Lopez", "Sanchez", "Jimenez", 
                  "Gomez", "Hernandez", "Torres", "Diaz", "Vargas", "Castro", "Rojas", "Fernandez", 
                  "Morales", "Ramirez", "Silva", "Mendoza", "Ortiz", "Ruiz", "Molina", "Reyes", "Gutierrez"]
    
    departments = ["Ventas", "Marketing", "IT", "Recursos Humanos", "Finanzas", "Operaciones", 
                   "Logistica", "Produccion", "Calidad", "Administracion", "Legal", "Compras", 
                   "Servicio al Cliente", "Investigacion y Desarrollo", "Contabilidad"]
    
    positions = ["Gerente", "Coordinador", "Analista", "Asistente", "Especialista", "Supervisor", 
                 "Director", "Jefe", "Auxiliar", "Tecnico", "Consultor", "Ingeniero", "Desarrollador", 
                 "Diseñador", "Contador", "Auditor", "Ejecutivo", "Representante"]
    
    cities = ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena", "Bucaramanga", 
              "Cucuta", "Santa Marta", "Pereira", "Manizales", "Ibague", "Pasto", "Villavicencio"]
    
    contract_types = ["Indefinido", "Fijo", "Obra o Labor", "Prestacion de Servicios", "Temporal", "Aprendizaje"]
    
    education_levels = ["Bachillerato", "Tecnico", "Tecnologia", "Universitario", "Especializacion", "Maestria", "Doctorado"]
    
    employment_status = ["Activo", "Inactivo", "Suspendido", "Vacaciones", "Licencia", "Incapacidad"]
    
    payment_frequency = ["Mensual", "Quincenal", "Semanal"]
    
    bank_names = ["Bancolombia", "Banco de Bogota", "Davivienda", "BBVA", "Scotiabank", "Banco Popular", 
                  "Banco AV Villas", "Banco Caja Social", "Nequi", "Daviplata"]
    
    shift_types = ["Diurno", "Nocturno", "Mixto", "Rotativo"]
    
    performance_ratings = ["Excelente", "Sobresaliente", "Satisfactorio", "Necesita Mejorar", "Insuficiente"]
    
    # Generamos columnas básicas con datos aleatorios
    df = df_base \
        .withColumn("first_name", expr(f"element_at(array{tuple(first_names)}, int(rand() * {len(first_names)}) + 1)")) \
        .withColumn("last_name", expr(f"element_at(array{tuple(last_names)}, int(rand() * {len(last_names)}) + 1)")) \
        .withColumn("age", expr("int(rand() * 42) + 18")) \
        .withColumn("gender", expr("case when rand() > 0.52 then 'M' when rand() > 0.05 then 'F' else null end")) \
        .withColumn("city", expr(f"element_at(array{tuple(cities)}, int(rand() * {len(cities)}) + 1)")) \
        .withColumn("department", expr(f"element_at(array{tuple(departments)}, int(rand() * {len(departments)}) + 1)")) \
        .withColumn("position", expr(f"element_at(array{tuple(positions)}, int(rand() * {len(positions)}) + 1)")) \
        .withColumn("contract_type", expr(f"element_at(array{tuple(contract_types)}, int(rand() * {len(contract_types)}) + 1)")) \
        .withColumn("education_level", expr(f"element_at(array{tuple(education_levels)}, int(rand() * {len(education_levels)}) + 1)")) \
        .withColumn("employment_status", expr(f"element_at(array{tuple(employment_status)}, int(rand() * {len(employment_status)}) + 1)")) \
        .withColumn("payment_frequency", expr(f"element_at(array{tuple(payment_frequency)}, int(rand() * {len(payment_frequency)}) + 1)")) \
        .withColumn("bank_name", expr(f"element_at(array{tuple(bank_names)}, int(rand() * {len(bank_names)}) + 1)")) \
        .withColumn("shift_type", expr(f"element_at(array{tuple(shift_types)}, int(rand() * {len(shift_types)}) + 1)")) \
        .withColumn("performance_rating", expr(f"element_at(array{tuple(performance_ratings)}, int(rand() * {len(performance_ratings)}) + 1)"))
    
    # Fechas con problemas de calidad
    df = df \
        .withColumn("hire_date", expr("date_add('2015-01-01', int(rand() * 3285))")) \
        .withColumn("birth_date", expr("date_sub(current_date(), age * 365 + int(rand() * 365))")) \
        .withColumn("contract_end_date", 
                    expr("""case 
                        when contract_type = 'Indefinido' then null
                        when contract_type = 'Fijo' then date_add(hire_date, int(rand() * 730) + 180)
                        when rand() > 0.3 then date_add(hire_date, int(rand() * 365) + 90)
                        else null end""")) \
        .withColumn("last_promotion_date", 
                    expr("case when rand() > 0.4 then date_add(hire_date, int(rand() * 1000)) else null end")) \
        .withColumn("last_evaluation_date", 
                    expr("case when rand() > 0.2 then date_add(hire_date, int(rand() * 365)) else null end"))
    
    # Salarios con inconsistencias
    df = df \
        .withColumn("education_multiplier", 
                    expr("""case 
                        when education_level = 'Bachillerato' then 0.8
                        when education_level = 'Tecnico' then 0.95
                        when education_level = 'Tecnologia' then 1.1
                        when education_level = 'Universitario' then 1.4
                        when education_level = 'Especializacion' then 1.7
                        when education_level = 'Maestria' then 2.1
                        when education_level = 'Doctorado' then 2.8
                        else 1.0 end""")) \
        .withColumn("base_salary", expr("int(rand() * 8000000) + 1500000")) \
        .withColumn("monthly_salary", expr("int(base_salary * education_multiplier)")) \
        .withColumn("transport_allowance", expr("case when monthly_salary < 2600000 then 140606 else 0 end")) \
        .withColumn("food_allowance", expr("int(rand() * 200000) + 50000")) \
        .withColumn("bonus_percentage", expr("round(rand() * 0.25, 2)")) \
        .withColumn("annual_bonus", expr("int(monthly_salary * bonus_percentage)")) \
        .withColumn("overtime_hours", expr("int(rand() * 30)")) \
        .withColumn("overtime_pay", expr("int((monthly_salary / 240.0) * overtime_hours * 1.25)")) \
        .withColumn("commission", expr("case when department in ('Ventas', 'Comercial') then int(rand() * 2000000) else 0 end")) \
        .withColumn("total_compensation", expr("monthly_salary + transport_allowance + food_allowance + overtime_pay + commission"))
    
    # Deducciones
    df = df \
        .withColumn("health_contribution", expr("int(monthly_salary * 0.04)")) \
        .withColumn("pension_contribution", expr("int(monthly_salary * 0.04)")) \
        .withColumn("solidarity_fund", expr("case when monthly_salary > 4000000 then int(monthly_salary * 0.01) else 0 end")) \
        .withColumn("loan_deduction", expr("case when rand() > 0.6 then int(rand() * 500000) else 0 end")) \
        .withColumn("other_deductions", expr("int(rand() * 100000)")) \
        .withColumn("total_deductions", expr("health_contribution + pension_contribution + solidarity_fund + loan_deduction + other_deductions")) \
        .withColumn("net_salary", expr("total_compensation - total_deductions"))
    
    # Información de contacto con problemas de calidad
    df = df \
        .withColumn("email_domain", expr("element_at(array('gmail.com', 'hotmail.com', 'outlook.com', 'yahoo.com', 'empresa.co', 'corporativo.com', null, ''), int(rand() * 8) + 1)")) \
        .withColumn("email", 
                    expr("""case 
                        when rand() > 0.08 and email_domain is not null and email_domain != '' 
                        then concat(lower(first_name), '.', lower(last_name), int(rand() * 100), '@', email_domain)
                        when rand() > 0.5 then concat(lower(first_name), '@', email_domain)
                        else null end""")) \
        .drop("email_domain")
    
    # Teléfonos con formatos inconsistentes (problema típico de zona de aterrizaje)
    df = df \
        .withColumn("phone_rand", expr("rand()")) \
        .withColumn("phone_base", expr("concat(string(int(rand() * 50) + 300), string(int(rand() * 9000000) + 1000000))")) \
        .withColumn("phone", 
                    expr("""case 
                        when phone_rand < 0.15 then concat('+57 ', phone_base)
                        when phone_rand < 0.30 then concat('57', phone_base)
                        when phone_rand < 0.50 then concat('(57) ', phone_base)
                        when phone_rand < 0.70 then phone_base
                        when phone_rand < 0.85 then concat(substring(phone_base, 1, 3), '-', substring(phone_base, 4, 10))
                        else null end""")) \
        .drop("phone_rand", "phone_base")
    
    # Números de identificación con problemas
    df = df \
        .withColumn("id_number", expr("concat(string(int(rand() * 90000000) + 10000000), case when rand() < 0.03 then 'X' else '' end)")) \
        .withColumn("account_number", expr("concat(string(bigint(rand() * 9000000000) + 1000000000), case when rand() < 0.05 then '-' else '' end)"))
    
    # Métricas de desempeño y asistencia
    df = df \
        .withColumn("years_experience", expr("int(datediff(current_date(), hire_date) / 365.25)")) \
        .withColumn("sick_days_taken", expr("int(rand() * 15)")) \
        .withColumn("vacation_days_taken", expr("int(rand() * 20)")) \
        .withColumn("vacation_days_available", expr("int(years_experience * 15)")) \
        .withColumn("absences_count", expr("int(rand() * 10)")) \
        .withColumn("late_arrivals_count", expr("int(rand() * 20)")) \
        .withColumn("training_hours_completed", expr("int(rand() * 100)")) \
        .withColumn("certifications_count", expr("int(rand() * 5)")) \
        .withColumn("projects_completed", expr("int(rand() * 25)")) \
        .withColumn("performance_score", expr("round(rand() * 5, 1)"))
    
    # Información adicional con nulos y problemas de calidad
    df = df \
        .withColumn("supervisor_id", expr(f"case when rand() > 0.1 then int(rand() * {cantidad}) + 1 else null end")) \
        .withColumn("emergency_contact_name", expr("case when rand() > 0.15 then concat('Contacto ', string(int(rand() * 1000))) else null end")) \
        .withColumn("emergency_contact_phone", expr("case when rand() > 0.2 then concat('300', string(int(rand() * 9000000) + 1000000)) else null end")) \
        .withColumn("has_dependents", expr("rand() > 0.5")) \
        .withColumn("dependents_count", expr("case when has_dependents then int(rand() * 4) + 1 else 0 end")) \
        .withColumn("marital_status", expr("element_at(array('Soltero', 'Casado', 'Divorciado', 'Viudo', 'Union Libre', null), int(rand() * 6) + 1)")) \
        .withColumn("blood_type", expr("element_at(array('O+', 'O-', 'A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', null), int(rand() * 9) + 1)")) \
        .withColumn("has_disabilities", expr("rand() > 0.92")) \
        .withColumn("remote_work_eligible", expr("case when department in ('IT', 'Marketing', 'Ventas') then rand() > 0.3 else rand() > 0.8 end"))
    
    # Campos para ejercicios de limpieza
    df = df \
        .withColumn("duplicate_flag", expr("case when rand() < 0.03 then true else false end")) \
        .withColumn("data_source", expr("element_at(array('HRIS', 'Payroll System', 'Manual Entry', 'Excel Import', 'Legacy System'), int(rand() * 5) + 1)")) \
        .withColumn("data_quality_score", expr("case when rand() < 0.1 then null else round(rand() * 100, 1) end")) \
        .withColumn("last_updated", expr("timestamp(date_add(hire_date, int(rand() * 365)))")) \
        .withColumn("updated_by", expr("element_at(array('admin', 'hr_user', 'system', 'payroll_user', null), int(rand() * 5) + 1)")) \
        .withColumn("is_verified", expr("rand() > 0.25")) \
        .withColumn("notes", expr("case when rand() < 0.7 then null else concat('Nota empleado #', string(employee_id)) end")) \
        .withColumn("termination_date", expr("case when employment_status = 'Inactivo' and rand() > 0.3 then date_add(hire_date, int(rand() * 2000)) else null end")) \
        .withColumn("termination_reason", expr("case when termination_date is not null then element_at(array('Renuncia Voluntaria', 'Despido', 'Mutuo Acuerdo', 'Jubilacion', 'Fin de Contrato', null), int(rand() * 6) + 1) else null end"))
    
    # Limpieza y reordenamiento final
    df = df.drop("education_multiplier", "base_salary")
    
    df = df.select(
        "employee_id", "first_name", "last_name", "id_number", "email", "phone", 
        "birth_date", "age", "gender", "marital_status", "has_dependents", "dependents_count",
        "blood_type", "has_disabilities", "city", "emergency_contact_name", "emergency_contact_phone",
        "hire_date", "contract_type", "contract_end_date", "employment_status", "termination_date", "termination_reason",
        "department", "position", "supervisor_id", "shift_type", "remote_work_eligible",
        "education_level", "years_experience", "last_promotion_date", "last_evaluation_date",
        "performance_rating", "performance_score", "projects_completed", "training_hours_completed", "certifications_count",
        "monthly_salary", "transport_allowance", "food_allowance", "bonus_percentage", "annual_bonus",
        "overtime_hours", "overtime_pay", "commission", "total_compensation",
        "health_contribution", "pension_contribution", "solidarity_fund", "loan_deduction", 
        "other_deductions", "total_deductions", "net_salary",
        "payment_frequency", "bank_name", "account_number",
        "sick_days_taken", "vacation_days_taken", "vacation_days_available", 
        "absences_count", "late_arrivals_count",
        "duplicate_flag", "data_source", "data_quality_score", "is_verified", 
        "last_updated", "updated_by", "notes"
    )
    
    return df

# Generamos 1 millón de registros de empleados con problemas de calidad
df_empleados = generate_employee_spark(1000000)
# display(df_empleados)

In [0]:
df_empleados.createOrReplaceTempView("empleados")
df_empleados.write.format("delta").mode("overwrite").saveAsTable("workspace.employees.tbl_empleados")